In [ ]:
# =============================================================================
# jobs/28_link_thresholds_to_gauges.py
# Link the Observed flood warning thresholds to the gauges the real-time import holds.
#
# RUNS IN THE RESTRICTED DEPLOYMENT. It reads the thresholds (restricted schema) and writes its table there.
# It reads the telemetry mapping (job 16) from the unrestricted schema. Nothing restricted is written to the
# unrestricted schema. Cadence: manual, after 16 or 17 changes.
#
# ROWS KEPT: Observed thresholds whose CatDesc is Flood Warning. Every Type (ACTCON, RES) and Purpose inside that is
# kept, as columns. Forecast rows, Forecaster Activity and Information are left out.
# THE LINK (rules in utils/threshold_link.py): Location = the mapping's FFFS_ID (case and leading zeros ignored),
# the Parameter's family (H.obs is level) = the measure's family, and the threshold's Unit (mALD, mAOD ...) must agree
# with the measure's unit_name. A measure whose unit disagrees is never linked: the row says unit_mismatch and names the
# units the gauge has. Read the profile this prints, then fill in the alias CSV for units the two sources word differently.
#
# OUTPUT: config.TBL_FW_THRESHOLD_LINKS (restricted schema), one row per threshold, overwritten in full.
#
# Parameters (widgets):
#   basis_label       the value that marks an observed threshold (default Observed)
#   catdesc_label     the CatDesc kept (default Flood Warning)
#   z_zero_is_missing  a z of 0 is taken as "not set" and not used to convert (default true)
#   aliases_csv       OPTIONAL two-column CSV, threshold_unit and measure_unit, for units that mean the same
#   basis_col, location_col, unit_col, parameter_col ...  the file's columns (blank = find them)
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql.types import StringType, StructField, StructType

import config
from utils.helpers import get_spark, utc_now, table_exists
import importlib
import utils.threshold_link as _helpers_module
importlib.reload(_helpers_module)       # a cluster keeps an imported module until Python restarts; this picks up the pulled version
from utils.threshold_link import link_summary, link_thresholds, prepare_aliases, prepare_thresholds

spark = get_spark()

dbutils.widgets.text("basis_label", "Observed", "Basis value that marks an observed threshold")
dbutils.widgets.text("catdesc_label", "Flood Warning", "CatDesc to keep")
dbutils.widgets.dropdown("z_zero_is_missing", "true", ["true", "false"], "Treat a z of 0 as not set")
dbutils.widgets.text("aliases_csv", config.THRESHOLD_UNIT_ALIASES_CSV, "Unit aliases CSV (optional)")
for name, label in (("basis_col", "Basis column (Observed or Forecast)"), ("location_col", "Location (FFFS_ID) column"),
                    ("unit_col", "Unit column"), ("parameter_col", "Parameter column"), ("value_col", "Value column"),
                    ("catdesc_col", "CatDesc column")):
    dbutils.widgets.text(name, "", label)

basis_label = dbutils.widgets.get("basis_label").strip() or "Observed"
catdesc_label = dbutils.widgets.get("catdesc_label").strip() or "Flood Warning"
aliases_csv = dbutils.widgets.get("aliases_csv").strip()
zero_z_is_missing = dbutils.widgets.get("z_zero_is_missing").strip().lower() != "false"
columns = {k: dbutils.widgets.get(w).strip() for k, w in (("basis", "basis_col"), ("location", "location_col"), ("unit", "unit_col"),
                                                         ("parameter", "parameter_col"), ("value", "value_col"), ("catdesc", "catdesc_col"))}
columns = {k: v for k, v in columns.items() if v}

for t in (config.TBL_EA_THRESHOLDS, config.TBL_TELEMETRY_MAPPING):
    if not table_exists(spark, t):
        raise Exception(f"{t} does not exist. Run 17 (thresholds) and 16 (telemetry mapping) first.")

In [ ]:
# =============================================================================
# THRESHOLDS: read, filter, and profile what is left
# =============================================================================

raw = spark.table(config.TBL_EA_THRESHOLDS).toPandas()
thr, rep = prepare_thresholds(raw, columns, basis_label, catdesc_label)
print(f"Thresholds table: {rep['rows']:,} rows. Columns used: {rep['columns']}")
print(f"Basis values in the file: {rep['basis_values']}")
print(f"CatDesc values in the file: {rep['catdesc_values']}")
print(f"Kept {rep['rows_kept']:,} {basis_label} / {catdesc_label} rows "
      f"({rep['dropped_no_location']:,} dropped for no Location, {rep['dropped_no_value']:,} for no numeric Value).")
print("\nUnits of the rows kept (these must agree with the measures' units, or be aliased):")
display(pd.Series(rep["units"], name="rows").rename_axis("Unit").reset_index())
print("Parameters of the rows kept (H is level, Q is flow):")
display(pd.Series(rep["parameters"], name="rows").rename_axis("Parameter").reset_index())
print("Types of the rows kept:", rep["types"])
z_zero = int((thr["z"] == 0).sum()) if "z" in thr else 0
print(f"\nRows with z = 0: {z_zero:,} of {len(thr):,}. "
      + ("They are treated as not set, so an mAOD threshold there is not converted." if zero_z_is_missing else "z = 0 is being used as a real gauge zero."))


In [ ]:
# =============================================================================
# LINK TO THE GAUGES, THEN SUMMARISE
# =============================================================================

# Only matched rows are a confirmed gauge-to-measure link: an ambiguous row also carries a measure_uri, one per candidate, but it is a guess.
# These are the same rows config.TBL_MEASURES_MAPPED (the view over the register) is built from; the table is read instead because it
# also holds each row's parameter family, which the link needs.
mapping = spark.table(config.TBL_TELEMETRY_MAPPING).where("match_status = 'matched' AND measure_uri IS NOT NULL").toPandas()
print(f"Telemetry mapping rows matched to a measure: {len(mapping):,}")

alias_map = prepare_aliases(None)          # the defaults: a bare "m" is taken as mALD when no measure is exactly mALD
if aliases_csv:
    try:
        alias_map = prepare_aliases(pd.read_csv(aliases_csv, dtype=str))
        print(f"Unit aliases read from {aliases_csv}: {len(alias_map):,} units")
    except FileNotFoundError:
        print(f"No alias file at {aliases_csv}: units are compared as written (case and spacing ignored), with a bare m taken as mALD, mASD or mAOD.")

linked = link_thresholds(thr, mapping, alias_map, zero_z_is_missing)
summary = link_summary(linked)
print("\nLink status:", summary["by_status"])
print("Links by how the unit matched:", linked["unit_match"].value_counts().to_dict())
conv = linked[linked["unit_match"] == "converted"]
if len(conv):
    print(f"\n{len(conv):,} thresholds were moved to the gauge's datum with z (mAOD minus z for a local measure, plus z the other way). "
          f"Check a sample against levels you know:")
    display(conv[["fffs_id", "unit", "value", "z_used", "unit_name", "value_in_measure_unit"]].sample(min(15, len(conv)), random_state=1))
if summary["unit_mismatches"]:
    print("\nUnit mismatches (threshold unit and the units the gauge has), most common first. Add pairs to the alias CSV if they mean the same:")
    display(pd.DataFrame(summary["unit_mismatches"]))
print(f"\nGauges (distinct FFFS_ID) with at least one linked threshold: "
      f"{linked.loc[linked['link_status'].isin(['linked', 'linked_chosen']), 'key'].nunique():,} of {linked['key'].nunique():,}")
display(linked.head(20))

In [ ]:
# =============================================================================
# WRITE: restricted schema, overwritten in full
# =============================================================================

out = linked.drop(columns=["key", "unit_key"]).copy()
out["linked_at"] = utc_now().isoformat()
out = out.astype(object).where(out.notna(), None)
for c in out.columns:
    out[c] = out[c].map(lambda v: None if v is None else str(v))
schema = StructType([StructField(c, StringType(), True) for c in out.columns])
(
    spark.createDataFrame(out, schema).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.TBL_FW_THRESHOLD_LINKS)
)
spark.sql(f"""
    COMMENT ON TABLE {config.TBL_FW_THRESHOLD_LINKS} IS
    'Observed flood warning thresholds (CatDesc Flood Warning) from the EA thresholds file, each linked to the real-time gauge measure
     that agrees on FFFS_ID (Location), parameter family and unit. link_status: linked, linked_chosen, unit_mismatch, no_measure,
     no_gauge or no_family; link_note says why. Values are text; value and target_lead_minutes convert with CAST. Restricted.'
""")
print(f"Written {len(out):,} rows to {config.TBL_FW_THRESHOLD_LINKS}")
dbutils.notebook.exit("success")